# Instrument profiles

Nominal Gaussian resolution matching with offline STIS, WFC3, NIRISS, NIRSpec, NIRCam and MIRI LRS profiles.
This synthetic example models neither throughput nor detector noise. The model
has real source coverage outside the final detector limits for convolution.


In [ ]:
import astropy.units as u
import numpy as np
import matplotlib.pyplot as plt
from speclib import Spectrum, get_profile, list_instrument_profiles

wave = np.geomspace(0.5, 6.0, 30000) * u.um
x = wave.to_value(u.um)
flux = np.ones(wave.size)
for center in (1.2, 3.5, 4.5):
    flux += 4 * np.exp(-0.5 * ((x - center) / 0.0004)**2)
unit = u.erg / (u.s * u.cm**2 * u.AA)
source = Spectrum(spectral_axis=wave, flux=flux * unit)
list_instrument_profiles()


## Coverage, resolving power and sampling

Every result is an ordinary `Spectrum`; the default preserves valid source samples.
`valid_intervals` declares missing coverage explicitly, without artificial flux
samples or treating detector regions as independent spectral orders.
Automatic centers are not detector pixels. Resampling integrates implicit bins;
coarse bin integration and sparse sampling can change apparent line widths.
Preserve fine native sampling or request more centers for width measurements.
Explicit centers and all their bin edges must stay within valid coverage.


In [ ]:
profile = get_profile("JWST/NIRSpec/PRISM/CLEAR")
print(profile.configuration)
print(profile.valid_intervals)
print(profile.calibration_version)
print(profile.resolution_definition)
print(profile.assumptions)
print(profile.provenance["url"])  # Original FITS, independent of coverage source.
print(profile.provenance["sha256"])
print(profile.coverage_source_url)
assert profile.provenance["filename"] == "jwst_nirspec_prism_disp.fits"

prism = source.observed_by(profile)
sampled = source.observed_by(profile, samples_per_resolution_element=3)
explicit_centers = np.linspace(0.65, 5.2, 300) * u.um
explicit = source.observed_by(profile, wavelength=explicit_centers)
assert len(prism.flux) > len(sampled.flux)
assert np.array_equal(explicit.wavelength, explicit_centers)
print(prism.valid_intervals)
continuous = prism


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(profile.wavelength, profile.resolving_power)
axes[0].set(xlabel="Wavelength (micrometers)", ylabel="Nominal resolving power", title="NIRSpec PRISM/CLEAR")
axes[1].plot(source.wavelength.to_value(u.um), source.flux.value, label="Source")
axes[1].plot(continuous.wavelength.to_value(u.um), continuous.flux.value, label="Broadened")
axes[1].plot(sampled.wavelength.to_value(u.um), sampled.flux.value, ".", label="3 samples/FWHM")
axes[1].set(xlim=(1.15, 1.25), xlabel="Wavelength (micrometers)", ylabel="Flux density (arbitrary units)")
axes[1].legend()
fig.tight_layout()
plt.show()


## Detector gaps

G395H/F290LP returns one ordinary spectrum with explicit `valid_intervals`.
Its arrays contain no samples in the 3.72–3.82 micrometer gap. Coverage is
approximate BOTS/S1600A1/SUB2048 geometry, not a visit-specific wavelength solution.
Source flux in the output gap still contributes to convolution. Plot each valid
interval separately; resampling and binning reject requests crossing the gap.

For a preloaded high-resolution `SpectralGrid` with sufficient source coverage,
`observed_grid = grid.observed_by("jwst/nirspec/g395h/f290lp")` matches every model.
Then `observed_grid.get_spectrum(teff, logg, metallicity)` returns an independent
`Spectrum` retaining the same detector coverage and gap-aware operations.
`get_flux` instead returns only the interpolated flux Quantity. See the
spectral-grids notebook for executable examples of both retrieval methods.


In [ ]:
gapped = source.observed_by("jwst/nirspec/g395h/f290lp")
print(gapped.valid_intervals)
fig, ax = plt.subplots(figsize=(7, 3), layout="constrained")
for lower, upper in gapped.valid_intervals:
    selected = (gapped.spectral_axis >= lower) & (gapped.spectral_axis <= upper)
    ax.plot(gapped.spectral_axis[selected].to_value(u.um), gapped.flux[selected].value)
ax.axvspan(3.72, 3.82, color="0.9", label="Detector gap")
ax.set(xlabel="Wavelength (micrometers)", ylabel="Flux density (arbitrary units)")
ax.legend()
plt.show()
assert not np.any((gapped.wavelength > 3.72*u.um) & (gapped.wavelength < 3.82*u.um))
centers = np.r_[np.linspace(3.0, 3.6, 20), np.linspace(4., 5., 20)]*u.um
resampled = gapped.resample(centers)
assert len(resampled.valid_intervals) == 2
selected = gapped.select_wavelength(4.*u.um, 5.*u.um)
assert len(selected.valid_intervals) == 1
try:
    gapped.resample([3.76, 3.77, 3.78]*u.um)
except ValueError as error:
    print(error)
else:
    raise AssertionError("Gap request was not rejected")
assert source.wavelength.min() == 0.5*u.um
assert "instrument_profile" not in source.meta


## Additional NIRSpec subarrays

Short NIRSpec identifiers continue to select SUB2048. Select a different regular
subarray explicitly as `jwst/nirspec/bots/s1600a1/<subarray>/<disperser>/<filter>`.
SUB1024A captures the outer portions on NRS1/NRS2; SUB1024B captures inner portions.
SUB512 and SUB512S share nominal wavelength coverage but have different spatial
heights (32 versus 16 pixels). All reuse the same nominal disperser R curve.
PRISM/CLEAR is unavailable with SUB1024A; multistripe subarrays are not bundled.
The user guide contains the complete supported coverage matrix.



In [ ]:
subarray_profile = get_profile("jwst/nirspec/bots/s1600a1/sub1024a/g395h/f290lp")
subarray_result = source.observed_by(subarray_profile)
print(subarray_profile.configuration, subarray_result.valid_intervals)
np.testing.assert_array_equal(subarray_result.valid_intervals.to_value(u.um),
                              [[2.87, 3.03], [4.51, 5.18]])
# The omitted middle region includes unread columns as well as the detector gap.
print("Physical detector gap:", subarray_profile.provenance["nominal_detector_gap_um"])
assert not np.any((subarray_result.wavelength > 3.03*u.um) &
                  (subarray_result.wavelength < 4.51*u.um))


## STIS, WFC3 and separate SOSS orders

STIS defaults to a centered point source through 52X2 at the prime CCD setting.
Its Gaussian widths are derived from model LSF half-height crossings and handbook
anchors. WFC3 uses nominal constant wavelength FWHM anchored to handbook reference
resolving powers. SOSS uses separate order dispersions and a two-pixel convention
from [Albert et al. (2023), Table 2 and Section 5.3](https://doi.org/10.1088/1538-3873/acd7a3).
All these prescriptions have wavelength-dependent R.

SOSS defaults to CLEAR/SUBSTRIP256. Order 2 uses practical coverage limited by
[current STScI calibration guidance](https://jwst-docs.stsci.edu/jwst-calibration-status/niriss-calibration-status/niriss-soss-calibration-status).
Each order returns an independent ordinary spectrum; overlapping coverage does
not perform order extraction, decontamination or stitching. The source extends
past every requested boundary to supply real convolution padding.


In [ ]:
optical_wave = np.geomspace(0.25, 3.0, 40000) * u.um
optical_source = Spectrum(spectral_axis=optical_wave,
                          flux=np.ones(optical_wave.size) * source.flux.unit)
new_modes = ["hst/stis/g430l", "hst/stis/g750l", "hst/wfc3/g102",
             "hst/wfc3/g141", "jwst/niriss/soss/order1", "jwst/niriss/soss/order2"]
matched = {}
for mode in new_modes:
    calibration = get_profile(mode)
    matched[mode] = optical_source.observed_by(calibration)
    assert type(matched[mode]) is Spectrum
    print(mode, calibration.quality, matched[mode].valid_intervals)
    print(calibration.provenance["equation"])

fig, ax = plt.subplots(figsize=(7, 3.5), layout="constrained")
for mode in new_modes:
    calibration = get_profile(mode)
    plot_wave = np.linspace(*calibration.valid_intervals[0].to_value(u.um), 300) * u.um
    ax.plot(plot_wave, calibration.resolving_power_at(plot_wave), label=mode)
ax.set(xlabel="Wavelength (micrometers)", ylabel="Nominal resolving power")
ax.legend(fontsize="small")


## NIRCam Module A Grism R time series

These four profiles describe nominal first-order point-source resolution, using the
[Greene et al. (2017) polynomial](https://doi.org/10.1117/1.JATIS.3.3.035001).
The shorthand selects `jwst/nircam/tsgrism/modulea/grismr/<filter>`; Grism C,
Module B and wide-field slitless configurations are unsupported.

The polynomial is confined to 2.4–5.0 µm. Speclib trims F277W's blue edge to
2.410 µm and F444W's red edge to 4.985 µm for safe convolution padding;
F322W2 and F356W retain the official 5% filter limits. These cuts differ from
filter edges and from F444W's approximate 5.03 µm TSO detector cutoff.
See the [STScI grisms reference](https://jwst-docs.stsci.edu/jwst-near-infrared-camera/nircam-instrumentation/nircam-grisms)
and [TSO coverage](https://jwst-docs.stsci.edu/jwst-near-infrared-camera/nircam-observing-modes/nircam-time-series-observations/nircam-grism-time-series).

The output remains an ordinary `Spectrum` (`SpectralGrid` for `grid.observed_by(...)`).
Native sampling is preserved unless requested otherwise; coarse output bins can
increase apparent FWHM. No throughput or second-order contamination model is applied.
Faint second-order spectra may occur with F277W/F322W2.


In [ ]:
nircam_wave = np.geomspace(2.39, 5.01, 24000) * u.um
nircam_flux = np.ones(nircam_wave.size)
for center in (2.7, 3.3, 3.6, 4.5):
    nircam_flux -= 0.7 * np.exp(-0.5 * ((nircam_wave.value-center)/0.00012)**2)
nircam_source = Spectrum(spectral_axis=nircam_wave, flux=nircam_flux*source.flux.unit)
fig, axes = plt.subplots(2, 2, figsize=(10, 6), layout="constrained")
for name, ax in zip(("f277w", "f322w2", "f356w", "f444w"), axes.flat):
    mode = "jwst/nircam/grismr/" + name
    result = nircam_source.observed_by(mode)  # Preserve native sampling.
    print(get_profile(mode).mode, result.valid_intervals)
    ax.plot(nircam_source.wavelength.to_value(u.um), nircam_source.flux.value,
            alpha=0.35, label="Input")
    ax.plot(result.wavelength.to_value(u.um), result.flux.value, label="Matched")
    ax.set(title=name.upper(), xlim=result.valid_intervals[0].to_value(u.um),
           xlabel="Wavelength (µm)", ylabel="Relative flux")
    ax.legend()

# Optional sampling uses the same API as every other profile.
f444w_sampled = nircam_source.observed_by(
    "jwst/nircam/grismr/f444w", samples_per_resolution_element=5)
f322w2_explicit = nircam_source.observed_by(
    "jwst/nircam/grismr/f322w2", wavelength=np.linspace(2.5, 4.0, 300)*u.um)
assert type(f444w_sampled) is Spectrum and type(f322w2_explicit) is Spectrum


## MIRI LRS slitless time series

`jwst/miri/lrs/slitless` selects conventional **P750L/SLITLESSPRISM** (canonical
`jwst/miri/lrs/slitless/slitlessprism`). The **nominal** curve is
R = 24 λ[µm] − 80: [STScI explicitly describes a linear trend](https://jwst-docs.stsci.edu/jwst-mid-infrared-instrument/miri-instrumentation/miri-filters-and-dispersers),
and [Kendrew et al. Figure 3](https://arxiv.org/abs/1512.03000) shows the design curve through 14 µm.
This is not a measured in-flight Gaussian LSF; λ/R is adopted as Gaussian FWHM.

The output interval is **5.5–12 µm**, with model support on **5–14 µm**.
5.5 µm leaves room for broad-kernel padding above the model boundary;
12 µm follows commonly useful coverage before falling red throughput.
[Foldover below about 4.5 µm](https://jwst-docs.stsci.edu/known-issues/miri-known-issues/miri-lrs-known-issues) is unblocked in slitless
mode and can push useful blue limits toward 5.2 µm for some spectra.
Neither cutoff guarantees useful data for a specific observation.

The [Cycle 6 IP/IPS subarrays](https://jwst-docs.stsci.edu/jwst-mid-infrared-instrument/miri-instrumentation/miri-detector-overview/miri-detector-subarrays) have different placements;
IPS also sacrifices red coverage. Both are initially unsupported, as are fixed
slit, WFSS and MRS. This synthetic continuum and absorption-band example does
not model throughput, foldover, detector effects or an actual stellar atmosphere.


In [ ]:
miri_wave = np.linspace(5.0, 14.0, 45001) * u.um
miri_flux = np.ones(miri_wave.size)
# Broad, shallow synthetic bands plus narrow lines demonstrate smoothing.
for center, depth, sigma in ((6.3, .06, .15), (8.0, .025, .4), (10.5, .2, .001)):
    miri_flux -= depth * np.exp(-.5 * ((miri_wave.value-center)/sigma)**2)
miri_source = Spectrum(spectral_axis=miri_wave, flux=miri_flux * source.flux.unit)
miri_profile = get_profile("jwst/miri/lrs/slitless")
miri = miri_source.observed_by(miri_profile)
miri_sampled = miri_source.observed_by(miri_profile, samples_per_resolution_element=5)
print(miri_profile.mode, miri_profile.quality, miri.valid_intervals)
fig, ax = plt.subplots(figsize=(8, 3), layout="constrained")
ax.plot(miri_wave, miri_flux, color="0.7", label="Synthetic input")
ax.plot(miri.wavelength.to_value(u.um), miri.flux.value, label="Native sampling")
ax.plot(miri_sampled.wavelength.to_value(u.um), miri_sampled.flux.value,
        ".", ms=2, label="5 centers/FWHM; integrated bins")
ax.set(xlim=(5.5, 12), xlabel="Wavelength (µm)", ylabel="Relative flux")
ax.legend()
plt.show()


## Calibration and limitations

The [NASA/STScI resolving-power tables](https://jwst-docs.stsci.edu/jwst-near-infrared-spectrograph/nirspec-instrumentation/nirspec-dispersers-and-filters)
use a 2.2-pixel resolution element, here interpreted as a Gaussian FWHM. The
[BOTS coverage table](https://jwst-docs.stsci.edu/jwst-near-infrared-spectrograph/nirspec-operations/nirspec-bots-operations/nirspec-bots-wavelength-ranges-and-gaps)
sets the output intervals. These nominal disperser curves do not reproduce
measured wide-slit point-source LSFs or non-Gaussian wings. Intrinsic source
widths are assumed negligible; sampling checks cannot establish physical model
resolution. See the instrument-profiles user guide for padding, resampling,
provenance and source-resolution requirements.

The [scientific provenance reference](../user_guide/instrument_provenance.rst)
links each original file and records its creation/history, conversion and source
limitations. `profile.provenance` exposes that manifest record offline. The
nominal tables are dispersion-derived ground/modeling prescriptions; adopting
their 2.2-pixel element as a Gaussian FWHM is Speclib's approximation, not an
empirical in-flight LSF measurement.

WFC3/UVIS G280 is deferred pending a supported wavelength-dependent spectral
FWHM prescription; it has no bundled alias. See the {ref}`G280 calibration review <g280-calibration-status>`.
